In [ ]:
import re
import json
import pandas as pd
from pathlib import Path

from tqdm import tqdm

In [ ]:
DATA_FOLDER = Path("../data")
OUTPUT_FOLDER = Path("../outputs")

In [ ]:
def fetch_candidates(df, file: str, pattern: str = r"^\d{3}-") -> pd.DataFrame:
    # Collect all new dataframes in a fast Python list
    new_dfs = []

    for subfolder in tqdm(OUTPUT_FOLDER.iterdir()):
        if not re.search(pattern, subfolder.name):
            print(f"Skipping subfolder {subfolder.name} as it does not match the pattern.")
            continue
        json_file = subfolder / file
        if not json_file.exists():
            print(f"Skipping subfolder {subfolder.name} as {file} does not exist.")
            continue
        with open(json_file, "r") as f:
            data = json.load(f)

        temp_df = pd.DataFrame(data)
        temp_df["run_name"] = subfolder.name
        new_dfs.append(temp_df)

    if new_dfs:
        return pd.concat([df] + new_dfs, ignore_index=True)

    return df


### Test Individual Runs

In [ ]:
import pandas as pd
import plotly.figure_factory as ff

test = pd.DataFrame()
test = fetch_candidates(
    test,
    file="candidate_passages_before_threshold.json",
    pattern=r"^[^\d]*$"
)
# test = fetch_candidates(
#     test,
#     file="candidate_passages_before_threshold.json",
#     pattern=r"\d+"
# )

plot_data = test.dropna(
    subset=["rerank_score", "topic", "retrieval_query"]
)

groups = [
    group["rerank_score"].to_numpy()
    for _, group in plot_data.groupby("retrieval_query")
]

labels = [
    str(label)
    for label, _ in plot_data.groupby("retrieval_query")
]

fig = ff.create_distplot(
    hist_data=groups,
    group_labels=labels,
    show_hist=False,
    show_rug=False,
    curve_type="kde"
)

fig.update_layout(
    width=1400,
    height=700,
    xaxis_title="Rerank Score",
    yaxis_title="Density",
    legend_title_text="Retrieval Query",
    template="plotly_white"
)

fig.add_vline(
    x=0.40,
    line_dash="dash",
    line_width=1,
    annotation_text="Threshold = 0.40"
)

fig.show()

In [51]:
import json

import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go
from scipy.stats import gaussian_kde


def load_run(run_name: str) -> pd.DataFrame:
    path = (
        OUTPUT_FOLDER
        / run_name
        / "candidate_passages_before_threshold.json"
    )

    with path.open(encoding="utf-8") as f:
        df = pd.DataFrame(json.load(f))

    df["run"] = run_name
    return df


# Configuration
run_names = [
    "migrants-worker-criminal",
    "migrants-worker-criminal-context",
]

threshold = 0.40
bill_title_column = "bill_title_de"


# Load data
plot_data = (
    pd.concat(
        [load_run(run_name) for run_name in run_names],
        ignore_index=True,
    )
    .dropna(subset=["rerank_score", "run"])
)


# KDE distributions
groups = [
    plot_data.loc[
        plot_data["run"].eq(run_name),
        "rerank_score",
    ].to_numpy()
    for run_name in run_names
]

fig = ff.create_distplot(
    hist_data=groups,
    group_labels=run_names,
    show_hist=False,
    show_rug=False,
    curve_type="kde",
)

# Layout
fig.add_vline(
    x=threshold,
    line_dash="dash",
    line_width=1,
    annotation_text=f"Threshold",
)

fig.update_layout(
    width=1400,
    height=700,
    xaxis_title="Rerank Score",
    yaxis_title="Density",
    legend_title_text="Reranking mode",
    template="plotly_white",
)

fig.show()